# 📦 Optimisation des Achats Multi-Références
## Réduction de la Valeur de Stock : 4 M€ → 2,5 M€
### Modèle MILP — Python · Pyomo · CBC

---

## Problématique

L'entreprise doit satisfaire sa demande mensuelle tout en **atteignant une cible de valeur de stock de 2,5 M€ en fin d'horizon fiscal** (actuellement à 4 M€). Les leviers disponibles sont : **quand passer une commande** et **quelle quantité commander**, en respectant les contraintes de taux de service et de quantité minimale de commande (MOQ).

---

## Formulation mathématique

### Indices

| Symbole | Signification |
|---|---|
| $i \in I = \{1,\ldots,n\}$ | Références (PN) |
| $t \in T = \{1,\ldots,m\}$ | Périodes (mois) |

### Paramètres

| Symbole | Signification |
|---|---|
| $D_{i,t}$ | Demande du PN $i$ en période $t$ |
| $p_i$ | Prix unitaire (€/unité) |
| $h_i = \frac{\tau}{12} \cdot p_i$ | Coût de possession mensuel (€/u/mois) |
| $f_i$ | Coût fixe de passation de commande (€) |
| $MOQ_i$ | Quantité minimale de commande |
| $LT_i$ | Délai de livraison (mois) |
| $S_{i,0}$ | Stock initial |
| $SS_i$ | Stock de sécurité (taux de service) |
| $TARGET$ | **Cible de valeur de stock en fin d'horizon (€)** |
| $M$ | Grand nombre (*big-M*) |

### Variables

| Variable | Domaine | Signification |
|---|---|---|
| $O_{i,t}$ | $\mathbb{R}^+$ | Quantité commandée lancée en début de $t$ |
| $S_{i,t}$ | $\mathbb{R}^+$ | Stock en fin de $t$ |
| $z_{i,t}$ | $\{0,1\}$ | 1 si commande passée pour PN $i$ en $t$ |

### Fonction objectif

$$\min\; Z = \sum_{i}\sum_{t} \left( h_i \cdot S_{i,t} + f_i \cdot z_{i,t} \right)$$

### Contraintes

$$\text{(C1 — Bilan)}\quad S_{i,t} = S_{i,t-1} + O_{i,\;t-LT_i} - D_{i,t} \quad \forall\,i,t$$

$$\text{(C2 — Service)}\quad S_{i,t} \geq SS_i \quad \forall\,i,t$$

$$\text{(C3 — MOQ min)}\quad O_{i,t} \geq MOQ_i \cdot z_{i,t} \quad \forall\,i,t$$

$$\text{(C4 — Big-M)}\quad O_{i,t} \leq M \cdot z_{i,t} \quad \forall\,i,t$$

$$\boxed{\text{(C5 — Cible fiscale)}\quad \sum_{i} p_i \cdot S_{i,m} \leq TARGET}$$

> **C5 est la contrainte centrale** : elle force la valeur totale du stock en fin de période $m$ à ne pas dépasser la cible de 2,5 M€.


---
## 1. Imports

In [ ]:
from pyomo.environ import *
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


---
## 2. Données du problème

**5 références, 6 mois** — stock initial total ≈ **4 000 000 €**, cible fin d'horizon ≤ **2 500 000 €**.


In [ ]:
# ════════════════════════════════════════════════════════
# INDICES
# ════════════════════════════════════════════════════════
I = ['PN-A', 'PN-B', 'PN-C', 'PN-D', 'PN-E']
T = [1, 2, 3, 4, 5, 6]
noms_t = {1:"Janv.", 2:"Févr.", 3:"Mars", 4:"Avr.", 5:"Mai", 6:"Juin"}

# ════════════════════════════════════════════════════════
# PARAMÈTRES PAR RÉFÉRENCE
# ════════════════════════════════════════════════════════
prix = {'PN-A': 120,  'PN-B': 15,   'PN-C': 350,  'PN-D': 8,    'PN-E': 200}

# Stock initial calibré pour que la valeur totale ≈ 4 000 000 €
# PN-A: 7 000 × 120 = 840 000 €
# PN-B: 47 000 × 15 = 705 000 €
# PN-C: 2 800 × 350 = 980 000 €
# PN-D: 94 000 × 8  = 752 000 €
# PN-E: 3 500 × 200 = 700 000 €
# TOTAL              = 3 977 000 € ≈ 4 M€
S0 = {'PN-A': 7000,  'PN-B': 47000, 'PN-C': 2800, 'PN-D': 94000, 'PN-E': 3500}

LT  = {'PN-A': 2,    'PN-B': 1,     'PN-C': 2,    'PN-D': 1,     'PN-E': 3}
MOQ = {'PN-A': 500,  'PN-B': 2000,  'PN-C': 100,  'PN-D': 5000,  'PN-E': 200}

f_cmd = {'PN-A': 400, 'PN-B': 150, 'PN-C': 800, 'PN-D': 100, 'PN-E': 600}

# ════════════════════════════════════════════════════════
# DEMANDE D[i,t]
# ════════════════════════════════════════════════════════
D = {
    ('PN-A',1):1200,('PN-A',2):1500,('PN-A',3):1000,
    ('PN-A',4):1700,('PN-A',5):1300,('PN-A',6):1600,
    ('PN-B',1):5000,('PN-B',2):4500,('PN-B',3):6000,
    ('PN-B',4):4000,('PN-B',5):5500,('PN-B',6):4800,
    ('PN-C',1): 350,('PN-C',2): 420,('PN-C',3): 300,
    ('PN-C',4): 500,('PN-C',5): 380,('PN-C',6): 450,
    ('PN-D',1):8000,('PN-D',2):7500,('PN-D',3):9000,
    ('PN-D',4):8500,('PN-D',5):9500,('PN-D',6):8000,
    ('PN-E',1): 250,('PN-E',2): 300,('PN-E',3): 220,
    ('PN-E',4): 350,('PN-E',5): 280,('PN-E',6): 320,
}

# ════════════════════════════════════════════════════════
# PARAMÈTRES CALCULÉS
# ════════════════════════════════════════════════════════
tau     = 0.20
h       = {i: round(tau/12 * prix[i], 4) for i in I}
SS      = {i: round(0.20 * sum(D[(i,t)] for t in T) / len(T)) for i in I}
M_bigM  = {i: sum(D[(i,t)] for t in T) for i in I}

# Cible fiscale
TARGET  = 2_500_000   # 2,5 M€ en fin de période 6

# ════════════════════════════════════════════════════════
# AFFICHAGE
# ════════════════════════════════════════════════════════
val_init = sum(prix[i]*S0[i] for i in I)

df_param = pd.DataFrame({
    'Prix (€/u)'    : prix,
    'Stock init (u)': S0,
    'Val. init (€)' : {i: prix[i]*S0[i] for i in I},
    'LT (mois)'     : LT,
    'MOQ (u)'       : MOQ,
    'f_cmd (€)'     : f_cmd,
    'h (€/u/mois)'  : {i: round(h[i],3) for i in I},
    'SS (u)'        : SS,
}).rename_axis("PN")

print(f"{'='*55}")
print(f"  Valeur initiale du stock  : {val_init:>14,.0f}  €")
print(f"  Cible fin d'horizon       : {TARGET:>14,.0f}  €")
print(f"  Réduction à atteindre     : {val_init-TARGET:>14,.0f}  €  "
      f"({(val_init-TARGET)/val_init*100:.1f} %)")
print(f"{'='*55}")
print()
display(df_param)

df_dem = pd.DataFrame(
    {noms_t[t]: {i: D[(i,t)] for i in I} for t in T}
).rename_axis("PN")
print("\nDemande D[i,t] (unités/mois) :")
display(df_dem)


---
## 3. Modèle MILP (Pyomo)

In [ ]:
m = ConcreteModel("Optimisation_Achats_Cible_Stock")

m.I = Set(initialize=I)
m.T = Set(initialize=T)

# ── Variables ───────────────────────────────────────────
m.O = Var(m.I, m.T, domain=NonNegativeReals)   # quantité commandée
m.S = Var(m.I, m.T, domain=NonNegativeReals)   # stock fin de période
m.z = Var(m.I, m.T, domain=Binary)             # décision de commande

print("✅  Variables :")
print(f"   O[i,t] : {len(I)*len(T)} continues  (quantités commandées)")
print(f"   S[i,t] : {len(I)*len(T)} continues  (stocks)")
print(f"   z[i,t] : {len(I)*len(T)} binaires   (décisions)")


In [ ]:
# ── Fonction objectif ───────────────────────────────────
m.obj = Objective(
    expr=sum(h[i]*m.S[i,t] + f_cmd[i]*m.z[i,t]
             for i in I for t in T),
    sense=minimize
)
print("✅  Objectif : min Σ(h_i × S[i,t] + f_i × z[i,t])")


In [ ]:
# ── C1 — Bilan de stock ─────────────────────────────────
def c1(m, i, t):
    S_prev    = S0[i] if t == 1 else m.S[i, t-1]
    t_cmd     = t - LT[i]
    reception = m.O[i, t_cmd] if t_cmd >= 1 else 0
    return m.S[i,t] == S_prev + reception - D[(i,t)]
m.C1 = Constraint(m.I, m.T, rule=c1)

# ── C2 — Taux de service : S[i,t] >= SS[i] ─────────────
def c2(m, i, t):
    return m.S[i,t] >= SS[i]
m.C2 = Constraint(m.I, m.T, rule=c2)

# ── C3 — MOQ minimum : O[i,t] >= MOQ_i × z[i,t] ───────
def c3(m, i, t):
    return m.O[i,t] >= MOQ[i] * m.z[i,t]
m.C3 = Constraint(m.I, m.T, rule=c3)

# ── C4 — Big-M : O[i,t] <= M × z[i,t] ─────────────────
def c4(m, i, t):
    return m.O[i,t] <= M_bigM[i] * m.z[i,t]
m.C4 = Constraint(m.I, m.T, rule=c4)

# ── C5 — Cible fiscale fin d'horizon ────────────────────
T_fin = max(T)
m.C5 = Constraint(
    expr=sum(prix[i]*m.S[i, T_fin] for i in I) <= TARGET
)

n_cst = sum(1 for _ in m.component_data_objects(Constraint, active=True))
print("✅  Contraintes :")
print(f"   C1 Bilan de stock       : {len(I)*len(T):3d}")
print(f"   C2 Taux de service      : {len(I)*len(T):3d}")
print(f"   C3 MOQ minimum          : {len(I)*len(T):3d}")
print(f"   C4 Big-M                : {len(I)*len(T):3d}")
print(f"   C5 Cible fiscale        :   1  ← Σ p_i·S[i,{T_fin}] ≤ {TARGET:,.0f} €")
print(f"   TOTAL                   : {n_cst:3d}")


---
## 4. Résolution

In [ ]:
solver   = SolverFactory('cbc')
resultat = solver.solve(m, tee=False)
status   = str(resultat.solver.termination_condition)

print("=" * 58)
print("  RÉSULTAT DE LA RÉSOLUTION")
print("=" * 58)
print(f"  Statut solveur  : {status}")
print(f"  Coût total Z*   : {value(m.obj):>14,.2f}  €")
print("=" * 58)


---
## 5. Résultats

In [ ]:
# ── Plan de commandes ────────────────────────────────────
rows_z, rows_o = [], []
for i in I:
    rz = {"PN": i}; ro = {"PN": i}
    for t in T:
        zi = int(round(value(m.z[i,t])))
        oi = value(m.O[i,t])
        rz[noms_t[t]] = "✅" if zi else "—"
        ro[noms_t[t]] = f"{oi:,.0f}" if zi else "—"
    rows_z.append(rz); rows_o.append(ro)

print("Décisions de commande z[i,t] :")
display(pd.DataFrame(rows_z).set_index("PN"))
print("\nQuantités commandées O[i,t] (unités) :")
display(pd.DataFrame(rows_o).set_index("PN"))


In [ ]:
# ── Stocks et valeur par période ─────────────────────────
rows_s, rows_v = [], []
for i in I:
    rs = {"PN": i, "t=0": S0[i]}
    rv = {"PN": i, "t=0": prix[i]*S0[i]}
    for t in T:
        s = round(value(m.S[i,t]))
        rs[noms_t[t]] = s
        rv[noms_t[t]] = prix[i]*s
    rows_s.append(rs); rows_v.append(rv)

df_s = pd.DataFrame(rows_s).set_index("PN")
df_v = pd.DataFrame(rows_v).set_index("PN")

print("Stock S[i,t] (unités, fin de période) :")
display(df_s)
print("\nValeur de stock p_i × S[i,t] (€) :")
display(df_v.applymap(lambda x: f"{x:,.0f}"))

# Valeur totale par période
val_totale = {"t=0": sum(prix[i]*S0[i] for i in I)}
for t in T:
    val_totale[noms_t[t]] = sum(prix[i]*round(value(m.S[i,t])) for i in I)

print("\nValeur totale du stock par période :")
df_tot = pd.DataFrame([val_totale], index=["TOTAL (€)"])
display(df_tot.applymap(lambda x: f"{x:,.0f}"))


In [ ]:
# ── Vérification C5 ──────────────────────────────────────
val_fin = sum(prix[i]*value(m.S[i, T_fin]) for i in I)
val_ini = sum(prix[i]*S0[i] for i in I)

print(f"  Valeur stock initiale (t=0)         : {val_ini:>14,.0f}  €")
print(f"  Valeur stock fin {noms_t[T_fin]:<6} (t={T_fin})   : {val_fin:>14,.0f}  €")
print(f"  Cible TARGET                        : {TARGET:>14,.0f}  €")
print(f"  Réduction obtenue                   : {val_ini-val_fin:>14,.0f}  €  "
      f"({(val_ini-val_fin)/val_ini*100:.1f} %)")
print()
if val_fin <= TARGET * 1.005:
    print("  ✅  Contrainte C5 respectée — Cible atteinte !")
else:
    print(f"  ❌  Écart : {val_fin-TARGET:,.0f} €")


In [ ]:
# ════════════════════════════════════════════════════════
# GRAPHIQUE 1 : Trajectoire de la valeur de stock totale
# ════════════════════════════════════════════════════════
periodes_ax = ["t=0"] + list(noms_t.values())
vals_totales = [sum(prix[i]*S0[i] for i in I)] +                [sum(prix[i]*round(value(m.S[i,t])) for i in I) for t in T]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(periodes_ax, vals_totales, marker='o', lw=2.5,
        color='steelblue', label='Valeur totale du stock')
ax.fill_between(periodes_ax, vals_totales, alpha=0.12, color='steelblue')
ax.axhline(sum(prix[i]*S0[i] for i in I), color='grey', ls=':',
           lw=1.5, label=f"Stock initial ({val_ini:,.0f} €)")
ax.axhline(TARGET, color='red', ls='--', lw=2.0,
           label=f"Cible TARGET ({TARGET:,.0f} €)")

ax.set_title("Trajectoire de la valeur totale du stock — Plan optimal",
             fontsize=12, fontweight='bold')
ax.set_xlabel("Période"); ax.set_ylabel("Valeur (€)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x/1e6:.2f} M€"))
ax.legend(loc='upper right'); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()


In [ ]:
# ════════════════════════════════════════════════════════
# GRAPHIQUE 2 : Évolution du stock par référence
# ════════════════════════════════════════════════════════
fig, axes = plt.subplots(len(I), 1, figsize=(11, 3.2*len(I)), sharex=True)
colors = ['steelblue','darkorange','seagreen','tomato','mediumpurple']

for idx, (i, ax) in enumerate(zip(I, axes)):
    stock_vals = [S0[i]] + [round(value(m.S[i,t])) for t in T]
    val_vals   = [prix[i]*s for s in stock_vals]
    per        = ["t=0"] + list(noms_t.values())

    ax2 = ax.twinx()
    ax2.fill_between(per, val_vals, alpha=0.10, color=colors[idx])
    ax2.set_ylabel("Valeur (€)", fontsize=8, color=colors[idx])
    ax2.tick_params(axis='y', labelsize=8, labelcolor=colors[idx])
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x/1e3:.0f}k"))

    ax.plot(per, stock_vals, marker='o', color=colors[idx], lw=2, label=f"Stock {i}")
    ax.axhline(SS[i], color='red', ls='--', lw=1.3, label=f"SS = {SS[i]:,} u")

    for t in T:
        if round(value(m.z[i,t])) == 1:
            oi = int(round(value(m.O[i,t])))
            ax.annotate(f"🛒{oi:,}u", xy=(noms_t[t], stock_vals[t]),
                        xytext=(0,12), textcoords='offset points',
                        ha='center', fontsize=8, color='green',
                        arrowprops=dict(arrowstyle='->', color='green', lw=1))

    ax.set_title(f"{i}  |  {prix[i]} €/u  |  LT={LT[i]} mois  |  MOQ={MOQ[i]:,} u  "
                 f"|  SS={SS[i]:,} u  |  Val. init={prix[i]*S0[i]:,.0f} €",
                 fontsize=9, fontweight='bold')
    ax.set_ylabel("Unités"); ax.legend(fontsize=8); ax.grid(alpha=0.25)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))

axes[-1].set_xlabel("Période")
fig.suptitle("Évolution des stocks par référence — Plan de commandes optimal",
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ════════════════════════════════════════════════════════
# RÉSUMÉ FINAL
# ════════════════════════════════════════════════════════
cout_h   = sum(h[i]*value(m.S[i,t]) for i in I for t in T)
cout_cmd = sum(f_cmd[i]*value(m.z[i,t]) for i in I for t in T)
n_cmd    = int(round(sum(value(m.z[i,t]) for i in I for t in T)))

df_resume = pd.DataFrame([
    {"PN": i,
     "Nb commandes": int(round(sum(value(m.z[i,t]) for t in T))),
     "Coût possession (€)": round(sum(h[i]*value(m.S[i,t]) for t in T)),
     "Coût passation (€)":  round(sum(f_cmd[i]*value(m.z[i,t]) for t in T)),
     "Val. stock initial (€)": prix[i]*S0[i],
     "Val. stock final (€)": round(prix[i]*value(m.S[i,T_fin]))}
    for i in I
]).set_index("PN")
df_resume.loc["TOTAL"] = df_resume.sum()
display(df_resume.applymap(lambda x: f"{x:,.0f}"))

print()
print("═"*60)
print("  RÉSUMÉ FINAL")
print("═"*60)
print(f"  Valeur stock initiale (t=0)     : {val_ini:>12,.0f}  €")
print(f"  Valeur stock fin {noms_t[T_fin]:<6} (t={T_fin})  : {val_fin:>12,.0f}  €")
print(f"  Cible TARGET                    : {TARGET:>12,.0f}  €")
print(f"  Réduction                       : {val_ini-val_fin:>12,.0f}  €  ({(val_ini-val_fin)/val_ini*100:.1f} %)")
print(f"  ─────────────────────────────────────────────────")
print(f"  Coût possession total           : {cout_h:>12,.0f}  €")
print(f"  Coût passation total            : {cout_cmd:>12,.0f}  €")
print(f"  COÛT TOTAL OPTIMAL Z*           : {value(m.obj):>12,.0f}  €")
print(f"  ─────────────────────────────────────────────────")
print(f"  Nombre de commandes passées     : {n_cmd}")
print(f"  Taux de service                 : ✅ Respecté")
print(f"  Cible fiscale C5                : {'✅ Atteinte' if val_fin<=TARGET*1.005 else '❌ Non atteinte'}")
print("═"*60)
